# Schwinger $\mathbb{Z}_2$ LGT: The Zero-Knowledge Extrapolation Paradox
---
This notebook applies the framework to a $1D$ $\mathbb{Z}_2$ Lattice Gauge Theory (the Staggered Schwinger Model) on $N=3$ matter sites. The Hamiltonian features a staggered fermion mass, background electric field, and spatially-varying gauge couplings $g_l$.

## The Zero-Knowledge Paradox
We construct a highly adversarial train/test split to demonstrate structural generalization. We populate the training set heavily with the fully-interacting system (all links active: `[True, True]`). We then attempt to predict the dynamics of a **partially-connected sub-graph** (`[True, False]`) that the ML model has never witnessed.

Because the Fourier coefficients $b_l(x)$ natively encode the underlying structural symmetries of $H(x, \alpha)$, Kernel Ridge Regression successfully tracks the dynamics of the unseen topology, demonstrating generalization driven by physics rather than data distribution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tqdm

from quantum_learning_dynamics import Experiment
from quantum_learning_dynamics.hamiltonians.schwinger import SchwingerZ2Model
from quantum_learning_dynamics.observables.library import LocalMagnetization

plt.rcParams.update({
    'figure.dpi': 120, 
    'axes.grid': True, 
    'grid.alpha': 0.3, 
    'font.family': 'sans-serif',
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

SEED = 7
T_DENSE = np.linspace(0.0, 3.0, 100)
T_VALS  = np.linspace(0.01, 3.0, 15)

### Step 1: Initialize the Lattice Gauge Theory
We use 3 matter sites, resulting in 5 interleaved qubits (Matter 0, Link 0, Matter 1, Link 1, Matter 2). The unknown parameters $d=2$ are the gauge couplings $g_0$ and $g_1$ on the links.

In [ ]:
model = SchwingerZ2Model(num_matter=3, mass=0.5, electric_field=1.0, g_range=(0.6, 1.4))
obs   = LocalMagnetization(num_qubits=model.num_qubits, site=0)

# Using r=5 with 2nd-order Trotter ensures very high accuracy even at t=3.0
R_STEPS = 5

print(f"System: {model.num_qubits} Qubits, {model.d} Unknown Gauge Couplings")
print(f"Upload Paulis required for D-Block Parity: {model.upload_paulis}")

### Step 2: Adversarial Train/Test Split
We force the model to learn primarily from the fully-connected lattice (`[True, True]`), and test its capacity to predict the dynamics of a severed link (`[True, False]`).

In [ ]:
rng = np.random.default_rng(SEED)
alpha_star = model.sample_alpha(rng)
print(f"Ground-Truth Gauge Couplings (g): {alpha_star}")

X_train = (
    [[True, True ]] * 20 +    # Over-represent the fully connected graph
    [[False, True]] * 10      # Inject minimal secondary pattern for rank
)
rng.shuffle(X_train)

test_state = [True, False]    # STRICTLY HELD-OUT: Never seen during training

print(f"Training Set: {len(X_train)} configurations")
print(f"Held-out Test State: {test_state}")

In [ ]:
# Compute dense exact trajectory for the smooth background line
O_mat = obs.to_sparse_pauli_op().to_matrix()
n = model.num_qubits
exact_traj = np.empty(len(T_DENSE))

for i, t in enumerate(T_DENSE):
    if t == 0.0:
        psi = np.zeros(2**n, dtype=complex); psi[0] = 1.0
    else:
        U = model.exact_unitary(test_state, alpha_star, float(t))
        psi = U[:, 0]
    exact_traj[i] = float(np.real(np.conj(psi) @ O_mat @ psi))

In [ ]:
# Compute PAC-Learned and Trotter dynamics across time
trotter_pts = np.empty(len(T_VALS))
pac_pts     = np.empty(len(T_VALS))

pbar = tqdm.tqdm(enumerate(T_VALS), total=len(T_VALS), desc="Computing PAC and Trotter points")

for j, tau in pbar:
    exp = Experiment(
        model=model, observable=obs, method='meshgrid_kernel',
        tau=float(tau), r_steps=R_STEPS, 
        trotter_order=2, # Explicitly use 2nd-order Symmetric Trotter
        kernel_alpha=1e-2, seed=SEED,
    )
    
    # Train via Meshgrid Kernel
    y_train = exp.compute_trotter_labels(X_train, alpha_star, float(tau), R_STEPS)
    B_train = exp._extract_features(X_train)
    exp.learner.fit(B_train, y_train)

    # Predict held-out topology
    B_test = exp._extract_features([test_state])
    pac_pts[j]     = float(exp.learner.predict(B_test)[0])
    trotter_pts[j] = float(exp.compute_trotter_labels([test_state], alpha_star, float(tau), R_STEPS)[0])
    
    # Calculate Exact value at this specific tau
    U = model.exact_unitary(test_state, alpha_star, float(tau))
    psi = U[:, 0]
    exact_val = float(np.real(np.conj(psi) @ O_mat @ psi))
    
    # Print cleanly without breaking the tqdm progress bar
    tqdm.tqdm.write(f"t={tau:5.2f} | Exact: {exact_val:+.4f} | Trotter: {trotter_pts[j]:+.4f} | PAC: {pac_pts[j]:+.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(T_DENSE, exact_traj, label=r'Exact Continuous ($e^{-iHt}$)', 
         color='#a8cce3', linewidth=4)
ax.plot(T_VALS, trotter_pts, label=f'Exact Trotter Circuit (r={R_STEPS})', 
         color='#1f78b4', linewidth=2)
ax.plot(T_VALS, pac_pts, label='PAC-Learned Prediction', 
         color='black', marker='o', linestyle='dashed', linewidth=2)

ax.set_xlabel('Evolution Time $t$')
ax.set_ylabel('$\langle Z_0(t) \rangle$ on Unseen Graph')
ax.set_title('Schwinger $\mathbb{Z}_2$: Zero-Knowledge Extrapolation Paradox\nModel tracks unseen dynamics (Test: [True, False] | Train mostly: [True, True])', fontsize=12)
ax.legend(loc='best')
fig.tight_layout()
plt.show()